# 🛡️ Comment Moderation — 4-Level Toxicity Classifier
## Using Jigsaw Toxic Comment Classification Dataset

This notebook:
1. Downloads/prepares the Jigsaw Toxic Comment dataset from Hugging Face
2. Maps 6 toxicity labels → **4 moderation levels**
3. Trains a TF-IDF + Logistic Regression multi-class classifier
4. Evaluates with confusion matrix, classification report
5. Saves `model.pkl` and `vectorizer.pkl` for use in the Flask app

### 4 Moderation Levels
| Level | Code | Description |
|---|---|---|
| 0 | `CLEAN` | Safe to publish immediately |
| 1 | `MILD` | Slightly rude/borderline — auto-warn user |
| 2 | `TOXIC` | Toxic/obscene/insulting — hold for review |
| 3 | `SEVERE` | Threats / identity hate / severe abuse — auto-block |


In [ ]:
# Install required packages
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "datasets", "pandas", "scikit-learn", "numpy", "joblib",
    "matplotlib", "seaborn", "imbalanced-learn"])
print("✅ All packages installed")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import joblib
import re
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score)
from sklearn.pipeline import Pipeline
from sklearn.utils import resample

matplotlib.rcParams['figure.facecolor'] = '#0a0a1a'
matplotlib.rcParams['axes.facecolor'] = '#111128'
matplotlib.rcParams['text.color'] = '#e0e0ff'
matplotlib.rcParams['axes.labelcolor'] = '#e0e0ff'
matplotlib.rcParams['xtick.color'] = '#a0a0cc'
matplotlib.rcParams['ytick.color'] = '#a0a0cc'
matplotlib.rcParams['axes.edgecolor'] = '#333366'
matplotlib.rcParams['grid.color'] = '#222244'

print("✅ Imports done")


In [ ]:
# ─────────────────────────────────────────────────────────────
# Load Jigsaw Toxic Comment Classification dataset
# from Hugging Face (no Kaggle account needed)
# ─────────────────────────────────────────────────────────────
from datasets import load_dataset

print("📥 Loading Jigsaw dataset from Hugging Face...")
dataset = load_dataset("google/jigsaw_toxicity_pred", split="train")
df = dataset.to_pandas()

print(f"✅ Loaded {len(df):,} rows")
print(f"Columns: {list(df.columns)}")
df.head(3)


In [ ]:
# ─── Dataset overview ───
print("Shape:", df.shape)
print("\nLabel distribution:")
label_cols = ['toxic','severe_toxic','obscene','threat','insult','identity_hate']
print(df[label_cols].sum().sort_values(ascending=False))

# Percentage
print("\nPercentage of each label:")
pct = (df[label_cols].sum() / len(df) * 100).round(2)
print(pct)

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#7c6bff','#ff6b9d','#00e5c8','#ffc800','#ff8080','#88aaff']
bars = ax.bar(label_cols, df[label_cols].sum(), color=colors, edgecolor='none', width=0.6)
ax.set_title('Raw Label Distribution in Jigsaw Dataset', fontsize=14, pad=15, color='#e0e0ff')
ax.set_ylabel('Count', color='#a0a0cc')
ax.bar_label(bars, fmt='{:,.0f}', padding=4, color='#a0a0cc', fontsize=9)
plt.tight_layout()
plt.savefig('label_distribution.png', dpi=140, bbox_inches='tight')
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────
# Map 6 Jigsaw labels → 4 moderation levels
#
#  CLEAN  (0): No toxic labels at all
#  MILD   (1): toxic=1 but nothing severe/threatening
#  TOXIC  (2): obscene=1 OR insult=1 (with or without toxic)
#  SEVERE (3): severe_toxic=1 OR threat=1 OR identity_hate=1
#
# Priority: SEVERE > TOXIC > MILD > CLEAN
# ─────────────────────────────────────────────────────────────

def map_level(row):
    # Level 3 — SEVERE: threats, severe toxic, identity hate
    if row['severe_toxic'] == 1 or row['threat'] == 1 or row['identity_hate'] == 1:
        return 3
    # Level 2 — TOXIC: obscene or insulting language
    if row['obscene'] == 1 or row['insult'] == 1:
        return 2
    # Level 1 — MILD: labelled toxic but nothing else
    if row['toxic'] == 1:
        return 1
    # Level 0 — CLEAN
    return 0

df['moderation_level'] = df.apply(map_level, axis=1)

level_names = {0: 'CLEAN', 1: 'MILD', 2: 'TOXIC', 3: 'SEVERE'}
df['level_name'] = df['moderation_level'].map(level_names)

print("4-Level distribution:")
counts = df['moderation_level'].value_counts().sort_index()
for k, v in counts.items():
    print(f"  Level {k} ({level_names[k]:6s}): {v:>7,}  ({v/len(df)*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

level_colors = ['#00e578', '#7c6bff', '#ffa53d', '#ff4444']
labels_plot = [f"Level {k}\n{level_names[k]}" for k in sorted(counts.index)]

axes[0].bar(labels_plot, counts.values, color=level_colors, edgecolor='none', width=0.5)
axes[0].set_title('4-Level Moderation Distribution', fontsize=13, color='#e0e0ff')
axes[0].set_ylabel('Count')

axes[1].pie(counts.values, labels=labels_plot, colors=level_colors,
            autopct='%1.1f%%', pctdistance=0.8,
            textprops={'color':'#e0e0ff', 'fontsize':9},
            wedgeprops={'edgecolor':'#0a0a1a', 'linewidth':2})
axes[1].set_title('Level Proportions', fontsize=13, color='#e0e0ff')

plt.tight_layout()
plt.savefig('level_distribution.png', dpi=140, bbox_inches='tight')
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────
# Text Preprocessing
# ─────────────────────────────────────────────────────────────
def clean_text(text):
    if pd.isna(text): return ""
    text = str(text).lower()
    text = re.sub(r'https?://\S+', ' url ', text)          # URLs
    text = re.sub(r'\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b', ' ip ', text)  # IPs
    text = re.sub(r'[^a-z0-9\s!?.,:;]', ' ', text)        # Keep basic punct
    text = re.sub(r'(.)\1{3,}', r'\1\1', text)           # aaaa → aa
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['comment_text'].apply(clean_text)

# Feature engineering extras
df['text_len'] = df['comment_text'].str.len()
df['cap_ratio'] = df['comment_text'].apply(
    lambda x: sum(1 for c in str(x) if c.isupper()) / max(len(str(x)),1))
df['exclaim_count'] = df['comment_text'].str.count('!')
df['question_count'] = df['comment_text'].str.count('\?')

print("✅ Text cleaned")
print(f"Sample clean text:\n{df['clean_text'].iloc[0][:200]}")

# Show examples of each level
print("\n── Sample comments per level ──")
for lvl in range(4):
    sample = df[df['moderation_level'] == lvl]['comment_text'].iloc[0]
    print(f"\nLevel {lvl} ({level_names[lvl]}): {sample[:120]}...")


In [ ]:
# ─────────────────────────────────────────────────────────────
# Handle Class Imbalance
# Strategy: undersample CLEAN, oversample minority classes
# ─────────────────────────────────────────────────────────────
from sklearn.utils import resample

df_0 = df[df['moderation_level'] == 0]  # CLEAN  (huge)
df_1 = df[df['moderation_level'] == 1]  # MILD
df_2 = df[df['moderation_level'] == 2]  # TOXIC
df_3 = df[df['moderation_level'] == 3]  # SEVERE

TARGET = 15000  # samples per class for training

df_0_down = resample(df_0, n_samples=TARGET, random_state=42)
df_1_bal  = resample(df_1, n_samples=min(TARGET, len(df_1)), replace=len(df_1)<TARGET, random_state=42)
df_2_bal  = resample(df_2, n_samples=min(TARGET, len(df_2)), replace=len(df_2)<TARGET, random_state=42)
df_3_bal  = resample(df_3, n_samples=min(TARGET, len(df_3)), replace=len(df_3)<TARGET, random_state=42)

df_balanced = pd.concat([df_0_down, df_1_bal, df_2_bal, df_3_bal]).sample(frac=1, random_state=42)
print(f"Balanced dataset: {len(df_balanced):,} rows")
print(df_balanced['moderation_level'].value_counts().sort_index())


In [ ]:
# ─────────────────────────────────────────────────────────────
# Train / Test Split + TF-IDF Pipeline
# ─────────────────────────────────────────────────────────────
X = df_balanced['clean_text']
y = df_balanced['moderation_level']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")

# TF-IDF vectorizer: words + character n-grams
vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 3),          # unigrams, bigrams, trigrams
    analyzer='word',
    min_df=2,
    sublinear_tf=True,           # log(1+tf) for compression
    strip_accents='unicode',
    token_pattern=r'\b\w+\b'
)

print("\n🔄 Fitting TF-IDF vectorizer...")
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)
print(f"Vectorizer vocab size: {len(vectorizer.vocabulary_):,}")
print(f"Matrix shape: {X_train_tfidf.shape}")


In [ ]:
# ─────────────────────────────────────────────────────────────
# Train Logistic Regression Classifier
# ─────────────────────────────────────────────────────────────
print("🔄 Training Logistic Regression...")

model = LogisticRegression(
    C=5.0,
    max_iter=1000,
    solver='lbfgs',
    multi_class='multinomial',
    class_weight='balanced',    # handles any remaining imbalance
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_tfidf, y_train)
print("✅ Model trained!")

# Quick accuracy
y_pred = model.predict(X_test_tfidf)
acc = accuracy_score(y_test, y_pred)
f1  = f1_score(y_test, y_pred, average='macro')
print(f"\nAccuracy : {acc:.4f} ({acc*100:.2f}%)")
print(f"Macro F1  : {f1:.4f}")


In [ ]:
# ─────────────────────────────────────────────────────────────
# Evaluation: Classification Report + Confusion Matrix
# ─────────────────────────────────────────────────────────────
target_names = ['CLEAN (0)', 'MILD (1)', 'TOXIC (2)', 'SEVERE (3)']
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=target_names, yticklabels=target_names,
            linewidths=1, linecolor='#0a0a1a', ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Confusion Matrix — 4-Level Moderation', fontsize=13, pad=15, color='#e0e0ff')
ax.set_ylabel('True Label', color='#a0a0cc')
ax.set_xlabel('Predicted Label', color='#a0a0cc')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=140, bbox_inches='tight')
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────
# Probability Distribution Visualization
# ─────────────────────────────────────────────────────────────
y_proba = model.predict_proba(X_test_tfidf)

fig, axes = plt.subplots(1, 4, figsize=(14, 4), sharey=False)
level_colors = ['#00e578', '#7c6bff', '#ffa53d', '#ff4444']

for i, (col, name, color) in enumerate(zip(range(4), target_names, level_colors)):
    mask = y_test == i
    axes[i].hist(y_proba[mask, i], bins=30, color=color,
                 alpha=0.85, edgecolor='none')
    axes[i].set_title(name, fontsize=10, color=color)
    axes[i].set_xlabel('Predicted Probability', color='#a0a0cc', fontsize=8)
    axes[i].set_ylabel('Count', color='#a0a0cc', fontsize=8)

fig.suptitle('Predicted Probability Distribution per True Class', fontsize=13, color='#e0e0ff', y=1.02)
plt.tight_layout()
plt.savefig('probability_distributions.png', dpi=140, bbox_inches='tight')
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────
# Top Features per Class
# ─────────────────────────────────────────────────────────────
feature_names = vectorizer.get_feature_names_out()
n_top = 15

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
level_colors_list = ['#00e578', '#7c6bff', '#ffa53d', '#ff4444']

for i, (name, color) in enumerate(zip(target_names, level_colors_list)):
    coef = model.coef_[i]
    top_idx = np.argsort(coef)[-n_top:][::-1]
    top_feats = [feature_names[j] for j in top_idx]
    top_vals  = coef[top_idx]
    
    axes[i].barh(top_feats[::-1], top_vals[::-1], color=color, alpha=0.8, edgecolor='none')
    axes[i].set_title(f'{name}\nTop Features', fontsize=9, color=color)
    axes[i].tick_params(labelsize=7)

fig.suptitle('Top Predictive Features per Moderation Level', fontsize=13, color='#e0e0ff')
plt.tight_layout()
plt.savefig('top_features.png', dpi=140, bbox_inches='tight')
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────
# Save Model Artifacts
# ─────────────────────────────────────────────────────────────
import os
os.makedirs('model', exist_ok=True)

joblib.dump(model,      'model/moderation_model.pkl')
joblib.dump(vectorizer, 'model/tfidf_vectorizer.pkl')

print("✅ Saved:")
print("   model/moderation_model.pkl")
print("   model/tfidf_vectorizer.pkl")

# Save level metadata too
import json
meta = {
    "levels": {
        "0": {"name": "CLEAN",  "action": "auto_approve", "color": "#00e578",
              "description": "Safe to publish immediately"},
        "1": {"name": "MILD",   "action": "warn_user",    "color": "#7c6bff",
              "description": "Borderline — auto-approved with warning to user"},
        "2": {"name": "TOXIC",  "action": "hold_review",  "color": "#ffa53d",
              "description": "Toxic/obscene — held for admin review"},
        "3": {"name": "SEVERE", "action": "auto_block",   "color": "#ff4444",
              "description": "Threats/hate — auto-blocked, never published"}
    },
    "model": "TF-IDF (50k features, trigrams) + Logistic Regression (multinomial)",
    "classes": 4,
    "dataset": "Jigsaw Toxic Comment Classification (Google/Jigsaw, CC0)"
}
with open('model/meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print("   model/meta.json")


In [ ]:
# ─────────────────────────────────────────────────────────────
# Inference Demo — Try it out!
# ─────────────────────────────────────────────────────────────
level_names = {0: 'CLEAN', 1: 'MILD', 2: 'TOXIC', 3: 'SEVERE'}
level_actions = {
    0: ('✅ Auto-Approved', '#00e578'),
    1: ('⚠️  Approved with warning', '#7c6bff'),
    2: ('🔶 Held for admin review', '#ffa53d'),
    3: ('🚫 Auto-blocked', '#ff4444'),
}

test_comments = [
    "Great post! Really enjoyed reading this, very insightful.",
    "I kind of disagree with your point here, but interesting take.",
    "You are such an idiot, this is completely wrong garbage.",
    "If you post this again I will find you and hurt you.",
    "This is absolute BS, nobody cares what you think!!!",
    "Interesting perspective, I hadn't thought about it that way.",
    "People like you make me sick. You're disgusting.",
    "The methodology described here seems flawed in step 3.",
]

print(f"{'Comment':<55} {'Level':<10} {'Action'}")
print("-" * 90)

for comment in test_comments:
    cleaned = clean_text(comment)
    vec = vectorizer.transform([cleaned])
    pred = model.predict(vec)[0]
    proba = model.predict_proba(vec)[0]
    action, _ = level_actions[pred]
    conf = proba[pred]
    short = comment[:52] + "..." if len(comment) > 55 else comment
    print(f"{short:<55} {level_names[pred]:<10} {action}  ({conf:.0%})")


## ✅ Integration with Flask

The saved model files in `model/` are automatically used by `app.py`.

### How it works in the Flask app:
```python
# moderator.py loads the model once at startup:
model      = joblib.load('model/moderation_model.pkl')
vectorizer = joblib.load('model/tfidf_vectorizer.pkl')

def predict_level(text):
    cleaned = clean_text(text)
    vec = vectorizer.transform([cleaned])
    level = model.predict(vec)[0]
    proba = model.predict_proba(vec)[0]
    return int(level), float(proba[level])
```

### 4-Level Actions:
| Level | Name | Flask Action |
|---|---|---|
| 0 | CLEAN | `status='approved'` — published immediately |
| 1 | MILD | `status='approved'` — published + user warned |
| 2 | TOXIC | `status='flagged'` — held for admin review |
| 3 | SEVERE | `status='blocked'` — never published, user warned strongly |
